# SYNCHRONIZATION PRIMITIVES

### 1. join() method

calling the join() method on a thread ENSURES that the thread has terminated before any succeeding code is executed. It helps in maintaining order within your code.

In [ ]:
import threading
import time

def ourThread(i):
    print("Thread {} Started".format(i))
    time.sleep(i*2)
    print("Thread {} Finished".format(i))

def main():
    thread1 = threading.Thread(target=ourThread, args=(1,))
    thread1.start()
    print("Is thread 1 Finished?")
    thread2 = threading.Thread(target=ourThread, args=(2,))
    thread2.start()
    thread2.join()
    thread1.join()
    print("Stuff to do after thread2 has finished")
    
if __name__ == '__main__':
    main()


# NOTE: consider above code, you started thread2(), and then immediately after, joined it. You essentially paused the main thread until thread 2 began and ended. 
# This basically rendered your code single threaded again, you didn't gain any advantage in starting thread2, it might as well have been a simple serial call of the ourThread() function.

# this shows that you need to be careful with how you manage threads using the join() method..

### 2. Locks

Locks help you control access to shared resources in a program. Author used the 'single bathroom in a multi-tenant flat' example here which made sense.

In [ ]:
# PROGRAM WITHOUT LOCKS (Program goes nowhere because both threads are trying to decrement / increment the same value with no order whatsoever)

import threading
import time
import random

counter = 1

def workerA():
    global counter
    while counter < 1000:
        counter += 1
        print("Worker A is incrementing counter to {}".format(counter))
        sleepTime = random.randint(0,1)
        time.sleep(sleepTime)

def workerB():
    global counter
    while counter > -1000:
        counter -= 1
        print("Worker B is decrementing counter to {}".format(counter))
        sleepTime = random.randint(0,1)
        time.sleep(sleepTime)

def main():
    t0 = time.time()
    thread1 = threading.Thread(target=workerA)
    thread2 = threading.Thread(target=workerB)
    thread1.start()
    thread2.start()
    thread1.join()
    thread2.join()
    t1 = time.time()
    print("Execution Time {}".format(t1-t0))

if __name__ == '__main__':
    main()

# using locks, we can CONTROL how both threads access the shared value, make it deterministic.

In [ ]:
# PROGRAM WITH LOCKS (thread1 is started first, so it acquires the lock first, does its job, releases the lock, and then the next thread in line acquires it and does its job & so on)

import threading
import time
import random
counter = 1

lock = threading.Lock()

def workerA():
    global counter
    lock.acquire()
    try:
        while counter < 1000:
            counter += 1
            print("Worker A is incrementing counter to {}".format(counter))
            sleepTime = random.randint(0,1)
            time.sleep(sleepTime)
    finally:
        lock.release()

def workerB():
    global counter
    lock.acquire()
    try:
        while counter > -1000:
            counter -= 1
            print("Worker B is decrementing counter to {}".format(counter))
            sleepTime = random.randint(0,1)
            time.sleep(sleepTime)
    finally:
        lock.release()

def main():
    t0 = time.time()
    thread1 = threading.Thread(target=workerA)
    thread2 = threading.Thread(target=workerB)
    thread1.start()
    thread2.start()
    thread1.join()
    thread2.join()
    t1 = time.time()
    print("Execution Time {}".format(t1-t0))

if __name__ == '__main__':
    main()

### 3. RLocks (Re-entrant locks)

RLocks are regular locks, but with 'levels'. A thread that has acquired the rlock and acquire it again. This increments a counter in the rlock by 1. Other threads will have to wait for the counter = 0 to acquire the rlock.

In [ ]:
# how can rlocks be useful for us? 
# it can come in handy when you, for instance, want to have thread-safe access for a method within a class that accesses other class methods.

import threading
import time
class myWorker():
    def __init__(self):
        self.a = 1
        self.b = 2
        self.Rlock = threading.RLock()

    def modifyA(self):
        with self.Rlock:
            print("Modifying A : RLock Acquired:{}".format(self.Rlock._is_owned()))
            print("{}".format(self.Rlock))
            self.a = self.a + 1
            time.sleep(5)

    def modifyB(self):
        with self.Rlock:
            print("Modifying B : RLock Acquired:{}".format(self.Rlock._is_owned()))
            print("{}".format(self.Rlock))
            self.b = self.b - 1
            time.sleep(5)

    def modifyBoth(self):
        print(f"Before being acquired by anything : {self.Rlock}")
        with self.Rlock:
            print("Rlock acquired, modifying A and B")
            print("{}".format(self.Rlock))
            self.modifyA()
            self.modifyB()
        print("{}".format(self.Rlock))
            
workerA = myWorker()
workerA.modifyBoth()

### 4. Condition

A primitive that's like an 'if' statement within a thread based on a DIFFERENT thread's state..
e.g. say threadA does some work, and threadB evaluates results obtained from A's work. Then you can apply a condition in B to not execute until A is done.

Particularly useful in producer/consumer situations...

In [ ]:
# fn description of the condition.

def Condition(*args, **kwargs):  
    """Factory function that returns a new condition variable object.  A condition variable allows one or more threads to wait until they are  notified by another thread.  If the lock argument is given and not None, it must be a Lock or RLock  object, and it is used as the underlying lock. Otherwise, a new RLock object is created and used as the underlying lock."""
    pass # rest of the function defn

Example of a publish-subscribe system which makes use of this sync primitive...

In [ ]:
# publisher

import threading
from threading import Condition

class Publisher(threading.Thread):
    def __init__(self, integers, condition: Condition):
        self.condition = condition
        self.integers = integers
        threading.Thread.__init__(self)

    def run(self):
        while True:
            integer = random.randint(0,1000)
            self.condition.acquire() # thread sleeps if this call isn't favorable, if it is, it moves ahead
            print("Condition Acquired by Publisher: {}".format(self.name))
            self.integers.append(integer)
            self.condition.notify() # wakes one thread (n = 1 by default) that's waiting for this condition
            print("Condition Released by Publisher: {}".format(self.name)) # code bet. acquire() & release() is the CRITICAL SECTION.
            self.condition.release()
            time.sleep(1)

# subscriber

class Subscriber(threading.Thread):
    def __init__(self, integers, condition: Condition):
        self.integers = integers
        self.condition = condition
        threading.Thread.__init__(self)

    def run(self):
        while True:
            self.condition.acquire()
            print("Condition Acquired by Consumer: {}".format(self.name))
            while True:
                if self.integers:
                    integer = self.integers.pop()
                    print("{} Popped from list by Consumer: {}".format(integer, self.name))
                    break
                print("Condition Wait by {}".format(self.name))
                self.condition.wait()
            print("Consumer {} Releasing Condition".format(self.name)) # code bet. acquire() & release() is the CRITICAL SECTION.
            self.condition.release()

    # my simpler version of run() which DOES NOT work as well (i was curious about why there's while loops in this code, can't they be tackled by conditions)
    # def run(self):
    #     while True:
    #         self.condition.acquire()
    #         print("Condition Acquired by Consumer: {}".format(self.name))

    #         if self.integers:
    #             integer = self.integers.pop()
    #             print("{} Popped from list by Consumer: {}".format(integer, self.name))
    #         else:
    #             print("Condition Wait by {}".format(self.name))
    #             self.condition.wait()

    #         print("Consumer {} Releasing Condition".format(self.name)) # code bet. acquire() & release() is the CRITICAL SECTION.
    #         self.condition.release()

# why use perpetual while loops instead of just an if condition for critical section?
# ans: I'll answer the subscriber, which has 2 such nested loops.
# 1. outer loop: This one is mandatory because, the thread needs to sustain after a single consume operation. Remove the outer loop, and you'll see that after one pop(), there is no code left for the thread to execute after release. The thread will just.. die. Same reason why we have a while loop in publisher. publisher needs to keep appending elements to array, and not just die after the first append.

# 2. inner loop: this one because of optimization.
# consider case where len(arr) = 1, and publisher woke up > 1 subscribers (maybe n > 1 or it used notify_all()). Let there be 2 threads awake now, A and B. A tries to acquire lock -> succeeds -> checks array and sees element, then pops it -> releases lock & sleeps.... After this, B -> tries to acquire lock -> succeeds -> checks array but no element.. so it hits wait() (meaning it'll wake up on the next notify() call of the publisher). The call eventually comes, and thread B's code starts where it left off.

# now without inner loop, after the wait() returns, B will just release condition for no reason, without ever checking the array... it will take this up the next time it gets the condition...
# with while loop, after wait(), B will restart the while loop and check the array to find a number in it, thus not wasting its chance... This is optimal.

In [ ]:
def main():
    integers = []
    condition = threading.Condition()
    # Our Publisher
    pub1 = Publisher(integers, condition)
    pub1.start()
    # Our Subscribers
    sub1 = Subscriber(integers, condition)
    sub2 = Subscriber(integers, condition)
    sub1.start()
    sub2.start()
    ## Joining our Threads
    pub1.join()
    sub1.join()
    sub2.join()

if __name__ == '__main__':
    main()

There's a better way of writing those publisher / subscriber classes btw, or for that matter, any code which deals with locks. You do it with context managers.

In [ ]:
# publisher

import threading
from threading import Condition

class Publisher(threading.Thread):
    def __init__(self, integers, condition: Condition):
        self.condition = condition
        self.integers = integers
        threading.Thread.__init__(self)

    def run(self):
        while True:
            integer = random.randint(0,1000)
            with self.condition:
                print("Condition Acquired by Publisher: {}".format(self.name))
                self.integers.append(integer)
                self.condition.notify() # wakes one thread (n = 1 by default) that's waiting for this condition
                print("Condition Released by Publisher: {}".format(self.name)) # code bet. acquire() & release() is the CRITICAL SECTION.
            time.sleep(1)

# subscriber

class Subscriber(threading.Thread):
    def __init__(self, integers, condition: Condition):
        self.integers = integers
        self.condition = condition
        threading.Thread.__init__(self)

    def run(self):
        while True:
            with self.condition:
                print("Condition Acquired by Consumer: {}".format(self.name))
                while True:
                    if self.integers:
                        integer = self.integers.pop()
                        print("{} Popped from list by Consumer: {}".format(integer, self.name))
                        break
                    print("Condition Wait by {}".format(self.name))
                    self.condition.wait()
                print("Consumer {} Releasing Condition".format(self.name)) # code bet. acquire() & release() is the CRITICAL SECTION.